In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package wordnet to /home/cjboi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
ROOT = "/home/cjboi/MAST30034_Python/project2/"
DATA = ROOT + "data/tables/"


In [3]:
consumers = pd.read_csv(str(DATA + "tbl_consumer.csv") , sep = '|')

print(consumers.shape)
consumers_sorted = consumers.sort_values("name").reset_index(drop=True)
consumers.head()


(499999, 6)


,name,address,state,postcode,gender,consumer_id
0,Yolanda Williams,413 Haney Gardens Apt. 742,WA,6935,Female,1195503
1,Mary Smith,3764 Amber Oval,NSW,2782,Female,179208
2,Jill Jones MD,40693 Henry Greens,NT,862,Female,1194530
3,Lindsay Jimenez,00653 Davenport Crossroad,NSW,2780,Female,154128
4,Rebecca Blanchard,9271 Michael Manors Suite 651,WA,6355,Female,712975


In [4]:
DATA_DIR = "../data/tables/"
MERCHANT_DIR = str(DATA_DIR + "/tbl_merchants.parquet")
CONSUMER_DIR = str(DATA_DIR + "/consumer_user_details.parquet")



spark = (
    SparkSession.builder.appName("MAST30034 Project 2 BNPL")
    .config("spark.sql.repl.eagerVal.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", True)
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

merchants = spark.read.parquet(MERCHANT_DIR)
consumer_deets = spark.read.parquet(CONSUMER_DIR)





Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/05 15:51:15 WARN Utils: Your hostname, CJpooter, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/05 15:51:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/cjboi/miniconda3/envs/proj2_bnpl/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/05 15:51:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
def read_trans(path: Path):
    files = {}
    start = 
    pass

SyntaxError: invalid syntax (1729440121.py, line 3)

In [5]:
first_trans = spark.read.parquet("/home/cjboi/MAST30034_Python/project2/data/tables/transactions_20210228_20210827_snapshot/order_datetime=2021-02-28")
first_trans.show()
print(first_trans.count(), first_trans.columns)
cons_deets = spark.read.parquet("/home/cjboi/MAST30034_Python/project2/data/tables/consumer_user_details.parquet")
cons_deets.show()
print(cons_deets.count(), cons_deets.columns)


merchant_schema = StructType([
    StructField("name", StringType(), True),
    StructField("tags", StringType(), True),
    StructField("merchant_abn", LongType(), True)
])



+-------+------------+------------------+--------------------+
|user_id|merchant_abn|      dollar_value|            order_id|
+-------+------------+------------------+--------------------+
|      1| 28000487688|133.22689421562643|0c37b3f7-c7f1-48c...|
|  18485| 62191208634| 79.13140006851712|9e18b913-0465-4fd...|
|      1| 83690644458|30.441348317517228|40a2ff69-ea34-465...|
|  18488| 39649557865| 962.8133405407585|f4c1a5ae-5b76-40d...|
|      2| 80779820715| 48.12397733548124|cd09bdd6-f56d-489...|
|  18489| 43186523025| 98.14878546968934|9008a98e-1b02-4de...|
|      3| 29566626791| 46.33087226118639|26b7574e-81c2-455...|
|  18490| 93558142492|232.83335268750145|2bda0665-796f-4f2...|
|      3| 32361057556| 87.34942171371054|633a7656-2fcc-4b8...|
|  18491| 64974914166|130.12601873970038|4bc15338-83eb-43d...|
|      5| 83177825742| 66.66426160206629|43e3b3fe-791b-47f...|
|  18493| 43186523025| 69.75645555605993|6c494428-b293-459...|
|      6| 21772962346|113.04510652600983|000ca5ac-3247-

In [6]:
# Read all 3 transaction folders
first =  spark.read.parquet(DATA + "transactions_20210228_20210827_snapshot/").withColumn("_source_order", F.lit(0))
second = spark.read.parquet(DATA + "transactions_20210828_20220227_snapshot/").withColumn("_source_order", F.lit(1))
third =  spark.read.parquet(DATA + "transactions_20220228_20220828_snapshot/").withColumn("_source_order", F.lit(2))

transactions = first.unionByName(second).unionByName(third).orderBy("_source_order")

before = transactions.count()

transactions = transactions.dropDuplicates(subset = ["order_id"])

after = transactions.count()

In [7]:

# No duplicates in transactions
stats = [
    ("Length Before = ", before),
    ("Length After = ", after),
    ("Dropped Dupilcates = ", before - after)
]
print(30*"-")
for label, value in stats:
    print(f"{label:<21}{value:>9}")



------------------------------
Length Before =       14195505
Length After =        14195505
Dropped Dupilcates =         0


In [8]:
merchants.select("tags").show(5, truncate=False)
# merchants.printSchema()

parts = F.split(F.col("tags"), r"[\)\]]\s*,\s*[\(\[]")

merchants = (
    merchants
    .withColumn("keyword_phrase", F.trim(F.regexp_replace(parts.getItem(0), r"[\[\(\]\)]", "")))
    .withColumn("revenue_code", F.trim(F.regexp_replace(parts.getItem(1), r"[\[\(\]\)]", "")))
    .withColumn("take_rate_raw", F.trim(F.regexp_replace(parts.getItem(2), r"[\[\(\]\)]", "")))
    .withColumn("take_rate", F.regexp_extract(F.col("take_rate_raw"), r"([\d.]+)", 1).cast("double"))
)

merchants = merchants.withColumn(
    "keywords",
    F.split(F.lower(F.col("keyword_phrase")), r"\s*,\s*")
)

FILLER_WORDS = ['-', 'and', 'or', 'al', 'an']

# split up by commas 
unique_keywords = (
    merchants
    .select(F.explode("keywords").alias("keyword"))
    .filter(F.col("keyword") != "")
    .distinct()
)
# got some filler words in there, split up by spaces for individual words
unique_keywords = (
    unique_keywords
    .select(F.explode(F.split(F.col("keyword"), r"\s+")).alias("keyword")) 
    .filter(F.col("keyword") != "")
    .distinct()
    .sort(F.col("keyword"))
)

# now lemmatizing with nltk
lemmatizer = WordNetLemmatizer()
words = sorted(r["keyword"] for r in unique_keywords.collect())
singularised = {w: lemmatizer.lemmatize(w, pos='n') for w in words}

+-----------------------------------------------------------------------------------------------------------------+
|tags                                                                                                             |
+-----------------------------------------------------------------------------------------------------------------+
|((furniture, home furnishings and equipment shops, and manufacturers, except appliances), (e), (take rate: 0.18))|
|([cable, satellite, and otHer pay television and radio services], [b], [take rate: 4.22])                        |
|([jewelry, watch, clock, and silverware shops], [b], [take rate: 4.40])                                          |
|([wAtch, clock, and jewelry repair shops], [b], [take rate: 3.29])                                               |
|([music shops - musical instruments, pianos, and sheet music], [a], [take rate: 6.33])                           |
+-----------------------------------------------------------------------

In [9]:
FILLER_WORDS = {'-', 'and', 'or', 'al', 'an'}

word_occurrences = (
    merchants
    .select(F.explode("keywords").alias("keyword"))                          # split by comma
    .filter(F.col("keyword") != "")
    .select(F.explode(F.split(F.col("keyword"), r"\s+")).alias("keyword"))    # split each phrase into words
    .filter(F.col("keyword") != "")
)

keyword_counts = (
    word_occurrences
    .replace(singularised, subset=["keyword"])
    .filter(~F.col("keyword").isin(FILLER_WORDS))
    .groupBy("keyword")
    .count()
    .orderBy(F.desc("count"))
)

# word_occurrences.show(n=100)
# keyword_counts.show(n=100)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

clean_words_df = keyword_counts.select(F.col("keyword").alias("clean_word"))
dirty_words_df = unique_keywords.select(F.col("keyword").alias("dirty_word"))

pairs = dirty_words_df.crossJoin(clean_words_df).withColumn(
    "distance", F.levenshtein(F.col("dirty_word"), F.col("clean_word"))
)

window = Window.partitionBy("dirty_word").orderBy(F.col("distance").asc())

nearest_match = (
    pairs
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .select("dirty_word", "clean_word", "distance")
    .sort(F.col("distance"), ascending = False)
)

# nearest_match.show(100, truncate=False) 

from pyspark.sql import functions as F

# only trust close matches — eyeball the distance column and pick a sensible cutoff
mapping_rows = ( 
    nearest_match
    .filter((F.col("distance") <= 3) & ~F.col("dirty_word").isin(FILLER_WORDS)).collect()
)
mapping_dict = {r["dirty_word"]: r["clean_word"] for r in mapping_rows}

# turn the small python dict into a Spark map literal
mapping_expr = F.create_map([F.lit(x) for pair in mapping_dict.items() for x in pair])

merchants = merchants.withColumn(
    "keywords_clean",
    F.transform(F.col("keywords"), lambda word: F.coalesce(mapping_expr[word], word))
)

MERCHANTS IS CLEAN



Now, merging fraudulent transactions on transactions records


In [10]:
cons_fraud_schema = StructType([
    StructField("user_id", LongType(), True),
    StructField("order_datetime", TimestampType(), True),
    StructField("fraud_probability", DoubleType(), True)
])

merch_fraud_schema = StructType([
    StructField("merchant_abn", LongType(), True),
    StructField("order_datetime", TimestampType(), True),
    StructField("fraud_probability", DoubleType(), True)
])


consumers = spark.read.parquet(str(DATA + "consumer_user_details.parquet"))

consumer_fraud_p = (
    spark.read
    .schema(cons_fraud_schema)
    .option("header", "true")
    .csv(str(DATA + 'consumer_fraud_probability.csv'))    
)

merchant_fraud_p = (
    spark.read.schema(merch_fraud_schema)
    .option("header", "true")
    .csv(str(DATA + "merchant_fraud_probability.csv"))
)

merchant_fraud_p = merchant_fraud_p.withColumn("order_datetime", F.col("order_datetime").cast("date"))
consumer_fraud_p = consumer_fraud_p.withColumn("order_datetime", F.col("order_datetime").cast("date"))

def _profile(sdf, name = "DataFrame"):
    """
    Prints a quick profile of spark dataframes
    """
    width = 40
    n_rows = sdf.count()
    n_cols = len(sdf.columns)

    print(f"{f'{name}':-^{width}}")
    print(f"{'rows':<15}{n_rows:>{width - 15},}")
    print(f"{'columns':<15}{n_cols:>{width - 15}}")
    print('-' * width)
    for col_name, dtype in sdf.dtypes:
        print(f"{col_name:<25}{dtype:>{width - 25}}")
    print("=" * width)

_profile(merchant_fraud_p, "merchant fraud probability")
_profile(consumer_fraud_p, "consumer fraud probabiltiy")


# ensure the schema didn't null *too* many values 
merhcant_null_counts = merchant_fraud_p.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in merchant_fraud_p.columns]
)

consuemr_null_counts = consumer_fraud_p.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in consumer_fraud_p.columns]
)
    
# merhcant_null_counts.show()
# consuemr_null_counts.show()

# Schema worked well, all types seem to be behaving well, merging fraudulent transactions
# on transactions dataframe. 

transactions = (
    transactions
    .join(
        merchant_fraud_p.select(
            "merchant_abn", "order_datetime", 
            F.col("fraud_probability").alias("merchant_fraud_probability")
        ),
        on = ["merchant_abn", "order_datetime"],
        how = "left"
    )
    .join(
        consumer_fraud_p.select(
            "user_id", "order_datetime",
            F.col("fraud_probability").alias("consumer_fraud_probability")    
        ), 
        on = ["user_id", "order_datetime"],
        how = "left"
    )
)

transactions.show()
transactions.printSchema()
merchant_fraud_p.printSchema()
consumer_fraud_p.printSchema()

transactions.join(merchant_fraud_p, on=["merchant_abn", "order_datetime"], how="inner").count()

    

-------merchant fraud probability-------
rows                                 114
columns                                3
----------------------------------------
merchant_abn                      bigint
order_datetime                      date
fraud_probability                 double
-------consumer fraud probabiltiy-------
rows                              34,864
columns                                3
----------------------------------------
user_id                           bigint
order_datetime                      date
fraud_probability                 double


+-------+--------------+------------+------------------+--------------------+-------------+--------------------------+--------------------------+
|user_id|order_datetime|merchant_abn|      dollar_value|            order_id|_source_order|merchant_fraud_probability|consumer_fraud_probability|
+-------+--------------+------------+------------------+--------------------+-------------+--------------------------+--------------------------+
|   7895|    2021-04-08| 86578477987| 5.974748534923688|3ecdf6a1-1fcc-46c...|            0|                      NULL|                      NULL|
|     83|    2021-08-20| 46804135891| 40.88736687931136|0f5df50c-5f44-46c...|            0|                      NULL|                      NULL|
|  18574|    2021-08-20| 40252040480|110.20656881653261|62131b0f-adf8-42e...|            0|                      NULL|                      NULL|
|   8220|    2021-04-08| 73619221575| 384.8642488548684|ad6d18c6-fcb6-4a7...|            0|                      NULL|      

4094

In [11]:
transactions.filter(F.col("merchant_fraud_probability").isNotNull()).count()
transactions.filter(F.col("consumer_fraud_probability").isNotNull()).count()

80560

## Progress so far-- for later reference

- Transactions dataset contains no duplicates (based on order_id)
- Transactions now merged with $\mathbb{P}[\text{fraud}]$, #TODO: have a look at the structure of missing values to decipher impoutation (likely thorugh modelling)  

